In [ ]:
from __future__ import annotations

from dataclasses import dataclass


@dataclass
class RetrievedChunk:

    chunk_id: str
    material_id: str
    project_id: str
    page_number: int | None
    text: str
    score: float | None = None
    source_file_name: str | None = None


class RetrievalService:

    def __init__(
        self,
        database,
        vector_index_name: str = (
            "document_chunks_vector_index"
        ),
    ):
        self.database = database
        self.vector_index_name = vector_index_name

    # --------------------------------------------------------
    # VECTOR SEARCH
    # --------------------------------------------------------

    def search(
        self,
        query: str,
        project_id: str,
        user_id: str,
        limit: int = 5,
    ) -> list[RetrievedChunk]:

        from app.services.ai_service import (
            AIService,
        )

        ai_service = AIService(
            database=self.database
        )

        query_embedding = ai_service.embed_query(
            query
        )

        return self.vector_search(
            query_embedding=query_embedding,
            project_id=project_id,
            user_id=user_id,
            limit=limit,
        )

    def vector_search(
        self,
        query_embedding: list[float],
        project_id: str,
        user_id: str,
        limit: int = 5,
        num_candidates: int = 100,
    ) -> list[RetrievedChunk]:

        if not query_embedding:
            return []

        collection = self.database.collection(
            "document_chunks"
        )

        pipeline = [
            {
                "$vectorSearch": {
                    "index": self.vector_index_name,
                    "path": "embedding",
                    "queryVector": query_embedding,
                    "numCandidates": max(
                        num_candidates,
                        limit,
                    ),
                    "limit": limit,
                    "filter": {
                        "project_id": project_id,
                        "user_id": user_id,
                    },
                }
            },
            {
                "$project": {
                    "_id": 0,
                    "chunk_id": 1,
                    "material_id": 1,
                    "project_id": 1,
                    "page_number": 1,
                    "text": 1,
                    "source_file_name": 1,
                    "score": {
                        "$meta": "vectorSearchScore"
                    },
                }
            },
        ]

        try:
            documents = collection.aggregate(
                pipeline
            )
        except Exception as exc:

            # Give a useful error instead of hiding an Atlas
            # index/configuration problem.
            raise RuntimeError(
                "MongoDB Atlas Vector Search failed. "
                "Verify that the vector index "
                f"'{self.vector_index_name}' exists "
                "and its dimensions match the embedding model."
            ) from exc

        return [
            self._to_result(document)
            for document in documents
        ]

    # --------------------------------------------------------
    # FALLBACK PROJECT RETRIEVAL
    # --------------------------------------------------------

    def get_project_chunks(
        self,
        project_id: str,
        user_id: str,
        limit: int = 20,
    ) -> list[RetrievedChunk]:

        collection = self.database.collection(
            "document_chunks"
        )

        documents = collection.find(
            {
                "project_id": project_id,
                "user_id": user_id,
            },
            {
                "_id": 0,
            },
        ).sort(
            "chunk_index",
            1,
        ).limit(limit)

        return [
            self._to_result(document)
            for document in documents
        ]

    @staticmethod
    def _to_result(
        document: dict,
    ) -> RetrievedChunk:

        return RetrievedChunk(
            chunk_id=document.get(
                "chunk_id",
                document.get("id", ""),
            ),
            material_id=document.get(
                "material_id",
                "",
            ),
            project_id=document.get(
                "project_id",
                "",
            ),
            page_number=document.get(
                "page_number"
            ),
            text=document.get(
                "text",
                "",
            ),
            score=document.get(
                "score"
            ),
            source_file_name=document.get(
                "source_file_name"
            ),
        )
